<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-05-bigquery-ml/lesson-5.3-llm-in-sql/practice/GCP_Capstone_5.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 5.3 — LLM in SQL — AI.GENERATE, VECTOR_SEARCH, Embeddings

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup — Auth, BigQuery client & remote models

Run this first. It authenticates with Application Default Credentials, creates the BigQuery client and helper functions, and registers the Gemini + embedding remote models. Every exercise below depends on it.

**Prerequisite:** run Lesson 5.1 first so `rag_data.document_features` exists (BigQuery tables persist per project).

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

# Ensure this module's datasets exist (idempotent -- BigQuery never auto-creates them).
# NOTE: cells that read rag_data.document_features need Lesson 5.1 run first.
for _ds in ('rag_data', 'ml_models'):
    _d = bigquery.Dataset(f'{PROJECT_ID}.{_ds}'); _d.location = 'US'
    client.create_dataset(_d, exists_ok=True)

def run_query(sql):
    return client.query(sql).to_dataframe()

def run_ddl(sql):
    job = client.query(sql)
    job.result()
    print(f'Done: {job.num_dml_affected_rows or "OK"}')

print(f'Connected to {PROJECT_ID}')

In [ ]:
# Register the remote models used by the AI.* / ML.* functions below.
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.gemini_flash`
  REMOTE WITH CONNECTION DEFAULT
  OPTIONS (ENDPOINT = 'gemini-3.6-flash')
''')
print('Gemini model created')

run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.embed_005`
  REMOTE WITH CONNECTION DEFAULT
  OPTIONS (ENDPOINT = 'text-embedding-005')
''')
print('Embedding model created')

In [ ]:
# Fixture: build rag_data.doc_chunks (chunk_id / chunk_text / doc_type) from
# Lesson 5.1's rag_data.document_features, so every exercise below uses the same
# table and column names as the main lesson HTML.
run_ddl(f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.doc_chunks` AS
SELECT
  ROW_NUMBER() OVER () AS chunk_id,
  doc_id,
  title         AS chunk_text,
  document_type AS doc_type
FROM `{PROJECT_ID}.rag_data.document_features`
''')
print('doc_chunks fixture ready')

## Exercise 1: AI.GENERATE Summary

**Difficulty:** Easy

Summarize 10 document chunks with AI.GENERATE. Print chunk_id + summary.

1. SELECT AI.GENERATE(CONCAT('Summarize:', text)).result
2. Run on 10 rows with LIMIT
3. Verify summaries are coherent

In [ ]:
# Summarize document chunks with AI.GENERATE (Gemini runs per row, in SQL).
# endpoint => 'gemini-3.6-flash' pins the model; without it the scalar
# AI.GENERATE uses BigQuery's default endpoint, not our gemini_flash model.
results = run_query(f'''
SELECT
  chunk_id, chunk_text,
  AI.GENERATE(
    CONCAT('Summarize in one sentence:\n', chunk_text),
    endpoint => 'gemini-3.6-flash'
  ).result AS summary
FROM `{PROJECT_ID}.rag_data.doc_chunks`
LIMIT 10
''')
print(results)

## Exercise 2: Structured Entity Extraction

**Difficulty:** Easy

Use AI.GENERATE with output_schema to extract people, orgs, dates.

1. Add output_schema parameter with typed fields
2. Use .* EXCEPT(full_response, status)
3. Verify typed columns in results

In [ ]:
# output_schema turns the free-text response into typed columns.
# Same AI.GENERATE call as Ex 1, but constrained to a schema.
results = run_query(f'''
SELECT
  chunk_id,
  ai.* EXCEPT(full_response, status)
FROM (
  SELECT
    chunk_id,
    AI.GENERATE(
      CONCAT('Extract entities from this text:\n', chunk_text),
      endpoint => 'gemini-3.6-flash',
      output_schema => 'people ARRAY<STRING>, organizations ARRAY<STRING>, sentiment STRING'
    ) AS ai
  FROM `{PROJECT_ID}.rag_data.doc_chunks`
  LIMIT 10
)
''')
print(results)
print('\nColumn types:')
print(results.dtypes)

## Exercise 3: Generate Embeddings

**Difficulty:** Easy

Create chunk_embeddings table with ML.GENERATE_EMBEDDING. Verify 768-dim arrays.

1. CREATE MODEL REMOTE for text-embedding-005
2. ML.GENERATE_EMBEDDING with RETRIEVAL_DOCUMENT
3. Check ARRAY_LENGTH(embedding) = 768

In [ ]:
# (embed_005 model already created in Setup.) Embed chunks as RETRIEVAL_DOCUMENT.
run_ddl(f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.chunk_embeddings` AS
SELECT
  chunk_id, doc_id, chunk_text,
  ml_generate_embedding_result AS embedding,
  ml_generate_embedding_status AS status
FROM ML.GENERATE_EMBEDDING(
  MODEL `{PROJECT_ID}.ml_models.embed_005`,
  (SELECT chunk_id, doc_id, chunk_text, chunk_text AS content
   FROM `{PROJECT_ID}.rag_data.doc_chunks`),
  STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_DOCUMENT' AS task_type)
)
WHERE ml_generate_embedding_status = ''
''')
print('Embeddings generated')

# Verify dimensions (should be 768 for text-embedding-005)
result = run_query(f'''
SELECT chunk_id, ARRAY_LENGTH(embedding) AS dims
FROM `{PROJECT_ID}.rag_data.chunk_embeddings`
LIMIT 5
''')
print(result)

## Exercise 4: VECTOR_SEARCH Top-5

**Difficulty:** Medium

Embed a query with RETRIEVAL_QUERY. Run VECTOR_SEARCH. Print top-5 + distances.

1. Embed query with RETRIEVAL_QUERY task type
2. VECTOR_SEARCH with top_k=5, COSINE
3. ORDER BY distance

In [ ]:
# Embed the query with RETRIEVAL_QUERY, then find the 5 nearest chunks.
results = run_query(f'''
SELECT
  base.chunk_id, base.chunk_text, distance
FROM VECTOR_SEARCH(
  TABLE `{PROJECT_ID}.rag_data.chunk_embeddings`,
  'embedding',
  (SELECT ml_generate_embedding_result AS embedding
   FROM ML.GENERATE_EMBEDDING(
     MODEL `{PROJECT_ID}.ml_models.embed_005`,
     (SELECT 'machine learning model training' AS content),
     STRUCT('RETRIEVAL_QUERY' AS task_type))),
  top_k => 5,
  distance_type => 'COSINE'
)
ORDER BY distance
''')
print('=== Vector Search Results (lower distance = more similar) ===')
print(results)

## Exercise 5: Create Vector Index

**Difficulty:** Medium

CREATE VECTOR INDEX with IVF. Compare search time before vs after.

1. CREATE VECTOR INDEX with IVF, COSINE
2. Add STORING clause for key columns
3. Check INFORMATION_SCHEMA.VECTOR_INDEXES status

In [ ]:
# Build an IVF vector index. STORING lets VECTOR_SEARCH return chunk_id/chunk_text
# straight from the index without re-reading the base table.
#
# IMPORTANT: BigQuery does NOT populate a vector index until the base table has
# at least 5,000 rows. Below that threshold the index stays PENDING/empty,
# index_status never reaches ACTIVE, and VECTOR_SEARCH silently falls back to a
# brute-force scan. On this small fixture the index will therefore NOT go ACTIVE
# and the timing below reflects brute force, not IVF. To actually exercise IVF,
# grow chunk_embeddings past 5,000 rows first.
run_ddl(f'''
CREATE OR REPLACE VECTOR INDEX chunk_idx
ON `{PROJECT_ID}.rag_data.chunk_embeddings`(embedding)
STORING (chunk_id, chunk_text)
OPTIONS (
  index_type = 'IVF',
  distance_type = 'COSINE',
  ivf_options = '{{"num_lists": 100}}'
)
''')
print('Vector index requested (build is asynchronous; needs >= 5,000 rows to go ACTIVE)')

# Poll build status. coverage_percentage hits 100 when the index is ACTIVE.
status = run_query(f'''
SELECT index_name, index_status, coverage_percentage,
       total_storage_bytes
FROM `{PROJECT_ID}.rag_data.INFORMATION_SCHEMA.VECTOR_INDEXES`
WHERE table_name = 'chunk_embeddings'
''')
print(status)

In [ ]:
# Compare search time before vs after the index is ACTIVE.
# Re-run the Ex 4 search and time it. NOTE: with fewer than 5,000 rows the index
# never reaches ACTIVE, so this stays brute-force and the "after" time will not
# improve — the comparison only becomes meaningful once the table exceeds 5,000
# rows and index_status reports ACTIVE.
import time

search_sql = f'''
SELECT base.chunk_id, base.chunk_text, distance
FROM VECTOR_SEARCH(
  TABLE `{PROJECT_ID}.rag_data.chunk_embeddings`, 'embedding',
  (SELECT ml_generate_embedding_result AS embedding
   FROM ML.GENERATE_EMBEDDING(
     MODEL `{PROJECT_ID}.ml_models.embed_005`,
     (SELECT 'machine learning model training' AS content),
     STRUCT('RETRIEVAL_QUERY' AS task_type))),
  top_k => 5, distance_type => 'COSINE')
ORDER BY distance
'''

t0 = time.time()
_ = run_query(search_sql)
print(f'Search elapsed: {time.time() - t0:.2f}s '
      '(brute-force until the index reports ACTIVE at >= 5,000 rows, IVF-accelerated after)')

## Exercise 6: Bulk Classify 500 Docs

**Difficulty:** Medium

Use AI.GENERATE for zero-shot classification. Compare with Lesson 5.1 LOGISTIC_REG.

1. AI.GENERATE with classification prompt
2. Compare predicted vs actual document_type
3. Calculate accuracy vs LOGISTIC_REG from 5.1

In [ ]:
# Zero-shot classification with AI.GENERATE — no training, just a prompt.
# endpoint => 'gemini-3.6-flash' pins the model (default endpoint otherwise).
results = run_query(f'''
SELECT
  chunk_id, chunk_text,
  LOWER(TRIM(AI.GENERATE(
    CONCAT('Classify as exactly one of: research_paper, invoice, legal, or form.\n\nText: ', chunk_text),
    endpoint => 'gemini-3.6-flash'
  ).result)) AS predicted_type,
  doc_type AS actual_type
FROM `{PROJECT_ID}.rag_data.doc_chunks`
''')
print('=== LLM Classification vs Actual ===')
print(results[['chunk_id','chunk_text','predicted_type','actual_type']].head(20))

# Zero-shot accuracy — contrast with the trained LOGISTIC_REG from Lesson 5.1.
acc = (results['predicted_type'].str.contains('|'.join(
        results['actual_type'].dropna().unique()), na=False))
match = (results['predicted_type'] == results['actual_type']).mean()
print(f'\nExact-match accuracy (zero-shot LLM): {match:.1%}')
print('Compare against your Lesson 5.1 LOGISTIC_REG accuracy — the trained model')
print('usually wins on volume/cost, the LLM often wins on rare edge-case labels.')

## Exercise 7: RAG-in-SQL

**Difficulty:** Challenge

Build complete RAG: embed query, VECTOR_SEARCH, ML.GENERATE_TEXT with context.

1. Innermost: ML.GENERATE_EMBEDDING with RETRIEVAL_QUERY
2. Middle: VECTOR_SEARCH top_k=5
3. Outer: ML.GENERATE_TEXT with STRING_AGG context

In [ ]:
# Complete RAG in one statement: embed query -> vector search -> grounded answer.
result = run_query(f'''
SELECT ml_generate_text_llm_result AS answer
FROM ML.GENERATE_TEXT(
  MODEL `{PROJECT_ID}.ml_models.gemini_flash`,
  (SELECT CONCAT(
     'Answer using ONLY this context:\n',
     STRING_AGG(
       FORMAT('[Source %s] %s', base.chunk_id, base.chunk_text),
       '\n'),
     '\n\nQuestion: ', MAX(query.q)
   ) AS prompt
   FROM VECTOR_SEARCH(
     TABLE `{PROJECT_ID}.rag_data.chunk_embeddings`,
     'embedding',
     (SELECT ml_generate_embedding_result AS embedding, content AS q
      FROM ML.GENERATE_EMBEDDING(
        MODEL `{PROJECT_ID}.ml_models.embed_005`,
        (SELECT 'What documents are about machine learning?' AS content),
        STRUCT('RETRIEVAL_QUERY' AS task_type))),
     top_k => 5,
     distance_type => 'COSINE')),
  STRUCT(1024 AS max_output_tokens, 0.2 AS temperature,
        TRUE AS flatten_json_output))
''')
print('=== RAG Answer ===')
print(result.iloc[0]['answer'] if len(result) > 0 else 'No result')

## Exercise 8: ask_documind() Procedure

**Difficulty:** Challenge

Create stored procedure. Test with 5 diverse questions. Verify grounded answers.

1. CREATE PROCEDURE with user_question parameter
2. Wrap RAG-in-SQL inside the procedure
3. Test with CALL ask_documind('question')

In [ ]:
# Wrap the RAG-in-SQL pipeline in a reusable stored procedure.
run_ddl(f'''
CREATE OR REPLACE PROCEDURE `{PROJECT_ID}.ml_models.ask_documind`(
  user_question STRING)
BEGIN
  SELECT ml_generate_text_llm_result AS answer
  FROM ML.GENERATE_TEXT(
    MODEL `{PROJECT_ID}.ml_models.gemini_flash`,
    (SELECT CONCAT(
       'Answer from context only:\n',
       STRING_AGG(FORMAT('[%s] %s', base.chunk_id, base.chunk_text), '\n'),
       '\n\nQ: ', MAX(query.q)
     ) AS prompt
     FROM VECTOR_SEARCH(
       TABLE `{PROJECT_ID}.rag_data.chunk_embeddings`, 'embedding',
       (SELECT ml_generate_embedding_result AS embedding, content AS q
        FROM ML.GENERATE_EMBEDDING(
          MODEL `{PROJECT_ID}.ml_models.embed_005`,
          (SELECT user_question AS content),
          STRUCT('RETRIEVAL_QUERY' AS task_type))),
       top_k => 3, distance_type => 'COSINE')),
    STRUCT(1024 AS max_output_tokens, TRUE AS flatten_json_output));
END
''')
print('ask_documind() procedure created')

In [ ]:
# Test with 5 diverse questions.
questions = [
    'What are the ML topics?',
    'Which documents mention training data?',
    'Are there any legal or compliance documents?',
    'Summarize what the invoices cover.',
    'What is the overall theme of this corpus?',
]
for q in questions:
    print(f'\nQ: {q}')
    try:
        result = run_query(
            f"CALL `{PROJECT_ID}.ml_models.ask_documind`('" + q.replace("'", "''") + "')")
        print('A:', result.iloc[0]['answer'] if len(result) > 0 else '(no answer)')
    except Exception as e:
        print(f'Note: {e}')